# ADM Model Snapshot Analysis

Inspects Pega Adaptive Decision Manager (ADM) model snapshots and predictor binning data to evaluate offer viability, understand feature importance, and compare model behaviour over time.

**Data sources (read-only — not part of the live decision pipeline):**
- `data-model-snapshop.json` — one record per model version per snapshot time
- `data-Predictor-binning-snapshot.json` — one record per predictor bin

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from tqdm import tqdm

print("Imports OK")

In [ ]:
ADM_DIR = Path("../data/adm")
MODEL_FILE = ADM_DIR / "data-model-snapshop.json"
PREDICTOR_FILE = ADM_DIR / "data-Predictor-binning-snapshot.json"


def load_jsonl(filepath: Path, description: str = "Loading") -> list[dict]:
    with open(filepath, encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)
    records = []
    with open(filepath, encoding="utf-8") as f:
        for line in tqdm(f, total=total_lines, desc=description):
            records.append(json.loads(line))
    return records


print("Loading model snapshots...")
df_model_snapshots = pl.DataFrame(load_jsonl(MODEL_FILE, "Model snapshots"))
print(f"  {df_model_snapshots.shape[0]:,} records, {df_model_snapshots.shape[1]} columns")

print("Loading predictor binning snapshots...")
df_predictor_binning = pl.DataFrame(load_jsonl(PREDICTOR_FILE, "Predictor binning"))
print(f"  {df_predictor_binning.shape[0]:,} records, {df_predictor_binning.shape[1]} columns")

## 2. Dataset Inspection

Inspect the structure, schema, and cross-dataset relationships of the three loaded datasets before proceeding to analysis.

In [ ]:
# One record per model (or model version)
print("=" * 60)
print("MODEL SNAPSHOTS")
print("=" * 60)

print(f"\nShape: {df_model_snapshots.shape}")
print(f"\nColumns:\n{df_model_snapshots.columns}")

print("\n--- Sample record ---")
df_model_snapshots.head(1).to_dicts()[0]

In [ ]:
# --- Predictor Binning Snapshots ---
# Multiple records per predictor (one per bin)
print("=" * 60)
print("PREDICTOR BINNING SNAPSHOTS")
print("=" * 60)

print(f"\nShape: {df_predictor_binning.shape}")
print(f"\nColumns:\n{df_predictor_binning.columns}")

# Unique predictors
unique_predictors = df_predictor_binning["pyPredictorName"].unique()
print(f"\n--- Predictor Summary ---")
print(f"Unique predictors: {len(unique_predictors)}")
print(f"Sample predictors: {unique_predictors[:5]}")

# Bins per predictor
bins_per_predictor = (
    df_predictor_binning
    .group_by("pyPredictorName")
    .agg(pl.len().alias("num_bins"))
    .sort("num_bins", descending=True)
)
print(f"\n--- Bins per Predictor (top 10) ---")
bins_per_predictor.head(10)

In [ ]:
# --- Key differences between datasets ---
print("=" * 60)
print("DATASET RELATIONSHIP")
print("=" * 60)

# Model ID links both datasets
model_ids_model = df_model_snapshots["pyModelID"].unique().to_list() if "pyModelID" in df_model_snapshots.columns else []
model_ids_predictor = df_predictor_binning["pyModelID"].unique().to_list()

print(f"\nModel IDs in model snapshots: {len(model_ids_model)}")
print(f"Model IDs in predictor binning: {len(model_ids_predictor)}")

if model_ids_model:
    common = set(model_ids_model) & set(model_ids_predictor)
    print(f"Common model IDs: {len(common)}")

## 3. Feature Importance Analysis

Analyze the `pyFeatureImportance` field from the predictor binning snapshots to understand which predictors contribute most to model decisions. We examine the distribution of importance scores, rank predictors by aggregate importance, and compare contributions by predictor type (numeric vs. symbolic).

In [ ]:
### 3.1 Feature Importance Distribution
print("=" * 60)
print("FEATURE IMPORTANCE DISTRIBUTION")
print("=" * 60)

# Basic stats
importance_col = df_predictor_binning["pyFeatureImportance"]
print(f"\n--- Summary Statistics ---")
print(importance_col.describe())

# Distribution by importance ranges (using lit() for string literals)
print(f"\n--- Importance Range Distribution ---")
df_importance_cats = (
    df_predictor_binning
    .with_columns(
        pl.when(pl.col("pyFeatureImportance") == 0)
        .then(pl.lit("Zero"))
        .when(pl.col("pyFeatureImportance") < 0.01)
        .then(pl.lit("Low (0-0.01)"))
        .when(pl.col("pyFeatureImportance") < 0.1)
        .then(pl.lit("Medium (0.01-0.1)"))
        .otherwise(pl.lit("High (>0.1)"))
        .alias("importance_category")
    )
)
df_importance_cats.group_by("importance_category").agg(pl.len().alias("count")).sort("count", descending=True)

In [ ]:
### 3.2 Top Features by Importance
print("=" * 60)
print("TOP FEATURES BY IMPORTANCE")
print("=" * 60)

# Aggregate importance by predictor (sum across bins)
predictor_importance = (
    df_predictor_binning
    .group_by("pyPredictorName")
    .agg(
        pl.sum("pyFeatureImportance").alias("total_importance"),
        pl.len().alias("num_bins"),
        pl.first("pyType").alias("type"),
        pl.first("pyModelID").alias("model_id")
    )
    .sort("total_importance", descending=True)
)

print(f"\n--- Top 15 Predictors by Total Importance ---")
print(predictor_importance.head(15))

# Show non-zero importance predictors
print(f"\n--- Predictors with Non-Zero Importance ---")
non_zero = predictor_importance.filter(pl.col("total_importance") > 0)
print(f"Total: {len(non_zero)} predictors with importance > 0")
print(non_zero.head(10))

### 3.3 Feature Importance Visualisation

In [ ]:
# 1. Distribution histogram of feature importance
fig1 = px.histogram(
    df_predictor_binning.to_pandas(),
    x="pyFeatureImportance",
    nbins=50,
    title="Distribution of Feature Importance (per bin)",
    labels={"pyFeatureImportance": "Feature Importance"},
)
fig1.show()

# 2. Top 20 predictors by total importance
top20 = predictor_importance.head(20).to_pandas()
fig2 = px.bar(
    top20,
    x="total_importance",
    y="pyPredictorName",
    orientation="h",
    title="Top 20 Predictors by Total Importance",
    labels={"total_importance": "Total Importance", "pyPredictorName": "Predictor"},
)
fig2.update_layout(yaxis=dict(autorange="reversed"))
fig2.show()

# 3. Importance split by predictor type
type_importance = (
    df_predictor_binning
    .group_by("pyType")
    .agg(pl.sum("pyFeatureImportance").alias("total_importance"))
    .to_pandas()
)
fig3 = px.pie(type_importance, values="total_importance", names="pyType", title="Importance by Predictor Type")
fig3.show()

## 4. Offer Selection and Comparison

To select models for deeper analysis, we evaluate two candidate offers from the model snapshot data. For each offer we check five viability criteria: temporal coverage (≥20 snapshots), sufficient response volume, a positive learning signal, active predictors, and acceptable model performance.

### 4.1 OnlineFlightCheckinOpen (Service – FlightInfo)

The online flight check-in offer is a high-frequency service interaction. We filter its model snapshots and evaluate each viability criterion to confirm it is a strong candidate for surrogate modelling and explanation analysis.

In [ ]:
TARGET_ISSUE = "Service"
TARGET_GROUP = "FlightInfo"
TARGET_NAME = "OnlineFlightCheckinOpen"

offer_df = df_model_snapshots.filter(
    (pl.col("pyIssue") == TARGET_ISSUE) &
    (pl.col("pyGroup") == TARGET_GROUP) &
    (pl.col("pyName") == TARGET_NAME)
)

print("\n" + "=" * 60)
print(f"CHECKING OFFER: {TARGET_ISSUE} / {TARGET_GROUP} / {TARGET_NAME}")
print("=" * 60)
print(f"\nNumber of snapshot rows: {offer_df.height}")

if offer_df.height == 0:
    print("No model snapshots found for this offer.")
else:
    # Cast numeric columns safely
    offer_df = offer_df.with_columns([
        pl.col("pyResponseCount").cast(pl.Float64, strict=False),
        pl.col("pyPositives").cast(pl.Float64, strict=False),
        pl.col("pyNegatives").cast(pl.Float64, strict=False),
        pl.col("pyActivePredictors").cast(pl.Float64, strict=False),
        pl.col("pyTotalPredictors").cast(pl.Float64, strict=False),
        pl.col("pyPerformance").cast(pl.Float64, strict=False),
        pl.col("pySuccessRate").cast(pl.Float64, strict=False),
    ])

    print("\n--- Basic overview ---")
    print(
        offer_df.select([
            pl.col("pyModelID").n_unique().alias("n_model_ids"),
            pl.col("pySnapshotTime").n_unique().alias("n_snapshots"),
            pl.col("pyResponseCount").max().alias("max_response_count"),
            pl.col("pyResponseCount").mean().alias("avg_response_count"),
            pl.col("pyPositives").max().alias("max_positives"),
            pl.col("pyPositives").sum().alias("sum_positives"),
            pl.col("pyActivePredictors").max().alias("max_active_predictors"),
            pl.col("pyTotalPredictors").max().alias("max_total_predictors"),
            pl.col("pyPerformance").max().alias("max_performance"),
            pl.col("pyPerformance").mean().alias("avg_performance"),
        ])
    )

    print("\n--- Snapshot progression (latest 10) ---")
    cols_to_show = [
        c for c in [
            "pySnapshotTime",
            "pyModelID",
            "pyResponseCount",
            "pyPositives",
            "pyNegatives",
            "pyActivePredictors",
            "pyTotalPredictors",
            "pyPerformance",
            "pySuccessRate",
        ] if c in offer_df.columns
    ]

    print(
        offer_df
        .sort("pySnapshotTime")
        .select(cols_to_show)
        .tail(10)
    )

    max_resp = offer_df["pyResponseCount"].max()
    max_pos = offer_df["pyPositives"].max()
    max_active = offer_df["pyActivePredictors"].max()
    n_snapshots = offer_df["pySnapshotTime"].n_unique()

    print("\n--- Decision checks ---")
    print(f"Enough snapshots (>20)? {'YES' if n_snapshots > 20 else 'NO'}  [{n_snapshots}]")
    print(f"Has learning signal (positives > 0)? {'YES' if (max_pos or 0) > 0 else 'NO'}  [{max_pos}]")
    print(f"Has active predictors (>0)? {'YES' if (max_active or 0) > 0 else 'NO'}  [{max_active}]")
    print(f"Reasonable response volume (>200)? {'YES' if (max_resp or 0) > 200 else 'NO'}  [{max_resp}]")

### 4.2 L5B15 Model (Sales – Luggage)

For comparison, we also evaluate the L5B15 luggage upsell offer — a purchase-driven model expected to show a lower positive rate and higher predictor complexity than the service-oriented check-in model.

In [ ]:
TARGET_ISSUE = "Sales"
TARGET_GROUP = "Luggage"
TARGET_NAME = "L5B15"

offer_df = df_model_snapshots.filter(
    (pl.col("pyIssue") == TARGET_ISSUE) &
    (pl.col("pyGroup") == TARGET_GROUP) &
    (pl.col("pyName") == TARGET_NAME)
)

print("\n" + "=" * 60)
print(f"CHECKING OFFER: {TARGET_ISSUE} / {TARGET_GROUP} / {TARGET_NAME}")
print("=" * 60)
print(f"\nNumber of snapshot rows: {offer_df.height}")

if offer_df.height == 0:
    print("No model snapshots found for this offer.")
else:
    # Cast numeric columns safely
    offer_df = offer_df.with_columns([
        pl.col("pyResponseCount").cast(pl.Float64, strict=False),
        pl.col("pyPositives").cast(pl.Float64, strict=False),
        pl.col("pyNegatives").cast(pl.Float64, strict=False),
        pl.col("pyActivePredictors").cast(pl.Float64, strict=False),
        pl.col("pyTotalPredictors").cast(pl.Float64, strict=False),
        pl.col("pyPerformance").cast(pl.Float64, strict=False),
        pl.col("pySuccessRate").cast(pl.Float64, strict=False),
    ])

    print("\n--- Basic overview ---")
    print(
        offer_df.select([
            pl.col("pyModelID").n_unique().alias("n_model_ids"),
            pl.col("pySnapshotTime").n_unique().alias("n_snapshots"),
            pl.col("pyResponseCount").max().alias("max_response_count"),
            pl.col("pyResponseCount").mean().alias("avg_response_count"),
            pl.col("pyPositives").max().alias("max_positives"),
            pl.col("pyPositives").sum().alias("sum_positives"),
            pl.col("pyActivePredictors").max().alias("max_active_predictors"),
            pl.col("pyTotalPredictors").max().alias("max_total_predictors"),
            pl.col("pyPerformance").max().alias("max_performance"),
            pl.col("pyPerformance").mean().alias("avg_performance"),
        ])
    )

    print("\n--- Snapshot progression (latest 10) ---")
    cols_to_show = [
        c for c in [
            "pySnapshotTime",
            "pyModelID",
            "pyResponseCount",
            "pyPositives",
            "pyNegatives",
            "pyActivePredictors",
            "pyTotalPredictors",
            "pyPerformance",
            "pySuccessRate",
        ] if c in offer_df.columns
    ]

    print(
        offer_df
        .sort("pySnapshotTime")
        .select(cols_to_show)
        .tail(10)
    )

    max_resp = offer_df["pyResponseCount"].max()
    max_pos = offer_df["pyPositives"].max()
    max_active = offer_df["pyActivePredictors"].max()
    n_snapshots = offer_df["pySnapshotTime"].n_unique()

    print("\n--- Decision checks ---")
    print(f"Enough snapshots (>20)? {'YES' if n_snapshots > 20 else 'NO'}  [{n_snapshots}]")
    print(f"Has learning signal (positives > 0)? {'YES' if (max_pos or 0) > 0 else 'NO'}  [{max_pos}]")
    print(f"Has active predictors (>0)? {'YES' if (max_active or 0) > 0 else 'NO'}  [{max_active}]")
    print(f"Reasonable response volume (>200)? {'YES' if (max_resp or 0) > 200 else 'NO'}  [{max_resp}]")

The L5B15 luggage model meets all viability criteria. Compared to the check-in model it shows a much lower positive rate (expected for a purchase offer), higher predictor complexity (up to 48 active predictors vs. ~24), and comparable average AUC (~0.70). The larger number of distinct model IDs and greater variability in predictor usage suggest a more heterogeneous model — making it a complementary case for studying explanation stability.

Both models are selected for further analysis.

In [ ]:
# --------------------------------------------------
# CONFIG: choose the two offers you want to compare
# --------------------------------------------------
OFFERS = [
    ("Service", "FlightInfo", "OnlineFlightCheckinOpen"),
    ("Sales", "Luggage", "L5B15"),
]

# --------------------------------------------------
# Helper: parse and prepare one offer
# --------------------------------------------------
def prepare_offer(df_model_snapshots, issue, group, name):
    df = df_model_snapshots.filter(
        (pl.col("pyIssue") == issue) &
        (pl.col("pyGroup") == group) &
        (pl.col("pyName") == name)
    )

    if df.height == 0:
        print(f"No rows found for {issue} / {group} / {name}")
        return None

    df = df.with_columns([
        pl.col("pyResponseCount").cast(pl.Float64, strict=False),
        pl.col("pyPositives").cast(pl.Float64, strict=False),
        pl.col("pyNegatives").cast(pl.Float64, strict=False),
        pl.col("pyActivePredictors").cast(pl.Float64, strict=False),
        pl.col("pyTotalPredictors").cast(pl.Float64, strict=False),
        pl.col("pyPerformance").cast(pl.Float64, strict=False),
        pl.col("pySuccessRate").cast(pl.Float64, strict=False),
        # Parse snapshot timestamp: keep first 15 chars like 20260415T015705
        pl.col("pySnapshotTime")
          .str.slice(0, 15)
          .str.strptime(pl.Datetime, format="%Y%m%dT%H%M%S", strict=False)
          .alias("snapshot_dt")
    ])

    # Aggregate per snapshot moment across model IDs
    out = (
        df.group_by("snapshot_dt")
        .agg([
            pl.col("pyResponseCount").sum().alias("response_count"),
            pl.col("pyPositives").sum().alias("positives"),
            pl.col("pyNegatives").sum().alias("negatives"),
            pl.col("pyActivePredictors").mean().alias("avg_active_predictors"),
            pl.col("pyTotalPredictors").mean().alias("avg_total_predictors"),
            pl.col("pyPerformance").mean().alias("avg_performance"),
            pl.col("pyModelID").n_unique().alias("n_models"),
        ])
        .sort("snapshot_dt")
        .with_columns([
            pl.lit(f"{issue} / {group} / {name}").alias("offer")
        ])
    )

    return out

# --------------------------------------------------
# Prepare both offers
# --------------------------------------------------
offer_frames = []
for issue, group, name in OFFERS:
    prepared = prepare_offer(df_model_snapshots, issue, group, name)
    if prepared is not None:
        offer_frames.append(prepared)

if not offer_frames:
    raise ValueError("No offer data found.")

plot_df = pl.concat(offer_frames).to_pandas()

# --------------------------------------------------
# Plot 1: Response count over time
# --------------------------------------------------
plt.figure(figsize=(10, 5))
for offer_name, group_df in plot_df.groupby("offer"):
    plt.plot(group_df["snapshot_dt"], group_df["response_count"], label=offer_name)
plt.title("Response count over time")
plt.xlabel("Snapshot time")
plt.ylabel("Response count")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# Plot 2: Positives over time
# --------------------------------------------------
plt.figure(figsize=(10, 5))
for offer_name, group_df in plot_df.groupby("offer"):
    plt.plot(group_df["snapshot_dt"], group_df["positives"], label=offer_name)
plt.title("Positive outcomes over time")
plt.xlabel("Snapshot time")
plt.ylabel("Positives")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# Plot 3: Average model performance over time
# --------------------------------------------------
plt.figure(figsize=(10, 5))
for offer_name, group_df in plot_df.groupby("offer"):
    plt.plot(group_df["snapshot_dt"], group_df["avg_performance"], label=offer_name)
plt.title("Average model performance over time")
plt.xlabel("Snapshot time")
plt.ylabel("Average performance")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# Plot 4: Average active predictors over time
# --------------------------------------------------
plt.figure(figsize=(10, 5))
for offer_name, group_df in plot_df.groupby("offer"):
    plt.plot(group_df["snapshot_dt"], group_df["avg_active_predictors"], label=offer_name)
plt.title("Average active predictors over time")
plt.xlabel("Snapshot time")
plt.ylabel("Average active predictors")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.3 Model Comparison Over Time

The four time-series plots reveal distinct dynamics between the two models:

1. **Response volume** — The L5B15 (luggage) model shows a strong increase in volume over time, eventually surpassing the check-in model, indicating growing exposure. The check-in model grows more steadily.

2. **Positive outcomes** — The OnlineFlightCheckinOpen model generates far more positive outcomes in absolute terms, reflecting its higher baseline engagement as a service interaction. The luggage model has fewer positives, consistent with a lower conversion rate typical of purchase offers.

3. **Model performance** — Both models maintain comparable AUC levels (~0.70–0.74) over time, indicating stable and effective learning. The check-in model shows a temporary spike then stabilizes; the luggage model improves more incrementally.

4. **Active predictor count** — The L5B15 model consistently uses more active predictors (growing from ~9 to ~26 over the window), indicating increasing model complexity. The check-in model stays simpler (~3–14 predictors), suggesting a more stable feature landscape.